# 07 — CW Diagnostics

## Purpose

Output continuous-wave (CW) tones on all quantum elements simultaneously so the entire RF chain can be verified with an external spectrum analyzer before proceeding to pulsed experiments.

## Methodology

1. **Select CW elements** — choose which quantum elements (qubit, readout, storage, etc.) to drive
2. **Run CW program** — the OPX outputs a non-stop CW tone on each selected element. The cell blocks while the program runs; interrupt the kernel when verification is complete.
3. **Manual SA verification** — use an external spectrum analyzer to confirm tone frequencies, power levels, and absence of spurs or harmonics

## Expected Outcomes

- Each element's tone appears at the expected frequency on the SA
- Power levels are consistent with configured LO power and element gain
- No unexpected spurious tones or harmonics visible
- RF chain connectivity confirmed for all elements

## Prerequisites

- **Notebook 00** — hardware session established
- **Notebook 01** — mixer calibration complete (important for clean tones)
- **Notebook 06** — coherence experiments completed (recommended, not required for CW)

## 1. Setup — Session Bootstrap

Open the notebook stage and load the prior coherence experiments checkpoint.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    continuous_wave,
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="07_cw_diagnostics",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

prev_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="06_coherence_experiments",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if prev_checkpoint is not None:
    print(f"Prior stage 06 status: {prev_checkpoint['status']}")

## 2. Configuration — CW Target Elements

Select which quantum elements to drive with continuous-wave tones. The default list includes all standard elements — qubit, readout, and storage.

In [ ]:
CW_ELEMENTS = ["qubit", "readout", "storage"]

print("CW target elements:")
for elem in CW_ELEMENTS:
    print(f"  - {elem}")

## 3. Execution — Run CW Program

Output continuous-wave tones on all configured elements for spectrum analyzer verification.

> **Note:** This cell blocks while the CW program is running. Stop the cell (interrupt kernel) when SA verification is complete.

In [ ]:
continuous_wave(session, elements=CW_ELEMENTS)

print("CW program stopped.")

## 4. Verification — Manual SA Checklist

Complete the following checks with the spectrum analyzer while the CW program is running:

- [ ] Qubit drive tone visible at expected frequency
- [ ] Readout tone visible at expected frequency
- [ ] Storage tone visible at expected frequency
- [ ] No unexpected spurs or harmonics
- [ ] Power levels match expected values

## 5. Checkpoint — Save CW Diagnostics Stage

Record the CW verification result. Since this is a manual verification step, no automated analysis metrics are saved.

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="07_cw_diagnostics",
    status="characterized",
    summary="CW diagnostics: continuous wave output verified on all elements via spectrum analyzer.",
    consumed_inputs={
        "cw_elements": CW_ELEMENTS,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage="08_pulse_waveform_definition",
    notes=[
        "Manual SA verification — no automated analysis.",
        "continuous_wave is a legacy proxy utility.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")